In [1]:
import openai
import os
from dotenv import load_dotenv, find_dotenv

# load local .env file containing the API key
_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv('OPENAI_API_KEY')

In [5]:
client = openai.OpenAI(
    api_key=openai.api_key,
    base_url="https://openrouter.ai/api/v1"
)

#### Example rate limit error

In [6]:
for _ in range(100):
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "Hello"}
        ],
        max_tokens = 10,
    )

#### Mitigate rate limit
##### Expotential backoff
1. Tenacity library

In [7]:
from tenacity import(
    retry,
    stop_after_attempt,
    wait_random_exponential
)

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def completion_with_backoff(**kwargs):
    return client.chat.completions.create(**kwargs)

completion_with_backoff(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])

ChatCompletion(id='gen-1774934848-Onm2jOVxMPHFC8h6h32s', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Once upon a time, in a lush, mystical forest filled with vibrant colors and enchanting sounds, there lived a young girl named Elara. She had a wild mane of curly hair that danced in the wind and curious green eyes that sparkled with wonder. Every morning, she would venture deeper into the woods, drawn by the whispers of adventure and the promise of magic.\n\nOne day, while exploring a hidden glade, Elara stumbled upon a shimmering pond that seemed to glow with an otherworldly light. As she approached, she noticed something unusual—a tiny, shimmering creature flitting above the water. It was a fairy, her delicate wings glimmering like fine crystal.\n\n“Hello there!” the fairy chimed, her voice like the tinkling of tiny bells. “I’m Liora, guardian of this pond. I need your help.”\n\nElara, a mix of excitement and curiosity swelling 

2. backoff library

In [9]:
import backoff

@backoff.on_exception(backoff.expo, openai.RateLimitError, max_time=60, max_tries=6)
def completions_with_backoff(**kwargs):
    return client.chat.completions.create(**kwargs)

completions_with_backoff(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])

ChatCompletion(id='gen-1774935119-JOfubu3pxV9NTomqWGJD', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='in a small, enchanted village nestled between rolling hills and a shimmering lake, there lived a kind-hearted girl named Elara. The village was known for its vibrant flowers and lush greenery, but it harbored a secret that only a few knew: a hidden portal to a magical realm.\n\nElara spent her days tending to her garden, where she grew the most beautiful blossoms in the village. She often dreamed of adventure and longed to explore the world beyond her familiar surroundings. Little did she know, her wish was about to come true.\n\nOne sunny afternoon, while collecting wildflowers by the forest\'s edge, Elara stumbled upon a peculiar, ancient stone archway covered in ivy. Drawn by an irresistible curiosity, she stepped closer. As she brushed away the leaves, a soft glow emanated from the arch, and a gentle breeze seemed to beckon he

3. manual 

In [ ]:
# imports
import random
import time

# define a retry decorator
def retry_with_exponential_backoff(
    func,
    initial_delay: float = 1,
    exponential_base: float = 2,
    jitter: bool = True,
    max_retries: int = 10,
    errors: tuple = (openai.RateLimitError,),
):
    """Retry a function with exponential backoff."""

    def wrapper(*args, **kwargs):
        # Initialize variables
        num_retries = 0
        delay = initial_delay

        # Loop until a successful response or max_retries is hit or an exception is raised
        while True:
            try:
                return func(*args, **kwargs)

            # Retry on specified errors
            except errors as e:
                # Increment retries
                num_retries += 1

                # Check if max retries has been reached
                if num_retries > max_retries:
                    raise Exception(
                        f"Maximum number of retries ({max_retries}) exceeded."
                    )

                # Increment the delay
                delay *= exponential_base * (1 + jitter * random.random())

                # Sleep for the delay
                time.sleep(delay)

            # Raise exceptions for any errors not specified
            except Exception as e:
                raise e

    return wrapper


@retry_with_exponential_backoff
def completions_with_backoff(**kwargs):
    return client.chat.completions.create(**kwargs)


completions_with_backoff(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])

#### Fallback model
If you encounter rate limit errors on your primary model, one option is to switch to a secondary model. This approach helps keep your application responsive when your primary model is throttled or unavailable.

However, fallback models can differ significantly in accuracy, latency, and cost. As a result, this strategy might not work for every use case; particularly those requiring highly consistent results.

In [ ]:
def completions_with_fallback(fallback_model, **kwargs):
    try:
        return client.chat.completions.create(**kwargs)
    except openai.RateLimitError:
        kwargs['model'] = fallback_model
        return client.chat.completions.create(**kwargs)

completions_with_fallback(fallback_model="gpt-4o", model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])

#### Reduce max_token to match expected completions
Rate limit usage is calculated based on the greater of:

1. max_tokens - the maximum number of tokens allowed in a response.
2. Estimated tokens in your input – derived from your prompt’s character count.

If you set max_tokens too high, your usage can be overestimated, even if the actual response is much shorter. To avoid hitting rate limits prematurely, configure max_tokens so it closely matches the size of the response you expect. This ensures more accurate usage calculations and helps prevent unintended throttling.

In [ ]:
def completions_with_max_tokens(**kwargs):
    return client.chat.completions.create(**kwargs)

completions_with_max_tokens(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}], max_tokens=100)

### Maximize throughput
If you're processing real-time requests from users, backoff and retry is a great strategy to minimize latency while avoiding rate limit errors.

However, if you're processing large volumes of batch data, where throughput matters more than latency, there are a few other things you can do in addition to backoff and retry.

#### Proactively adding delay between requests
If you are constantly hitting the rate limit, then backing off, then hitting the rate limit again, then backing off again, it's possible that a good fraction of your request budget will be 'wasted' on requests that need to be retried. This limits your processing throughput, given a fixed rate limit.

Here, one potential solution is to calculate your rate limit and add a delay equal to its reciprocal (e.g., if your rate limit 20 requests per minute, add a delay of 3–6 seconds to each request). This can help you operate near the rate limit ceiling without hitting it and incurring wasted requests.

In [ ]:
import time
# define a function that adds a delay to a Completion API call
def delayed_completion(delay_in_seconds: float = 1, **kwargs):
    # delay
    time.sleep(delay_in_seconds)

    # call api
    return client.chat.completions.create(**kwargs)

# calculate the required delay
rate_limit_per_min = 20
delay = 60.0 / rate_limit_per_min

delayed_completion(
    delay_in_seconds=delay,
    model="gpt-4o-mini",
    messages=[{"role":"user", "content":"Once upon a time,"}]
)

### Batching requests
The OpenAI API enforces separate limits for requests per minute/day (RPM/RPD) and tokens per minute (TPM). If you’re hitting RPM limits but still have available TPM capacity, consider batching multiple tasks into each request.

By bundling several prompts together, you reduce the total number of requests sent per minute, which helps avoid hitting the RPM cap. This approach may also lead to higher overall throughput if you manage your TPM usage carefully. However, keep the following points in mind:

- Each model has a maximum number of tokens it can process in one request. If your batched prompt exceeds this limit, the request will fail or be truncated.

- Batching can introduce extra waiting time if tasks are delayed until they’re grouped into a single request. This might affect user experience for time-sensitive applications.

- When sending multiple prompts, the response object may not return in the same order or format as the prompts that were submitted. You should try to match each response back to its corresponding prompt by post-processing the output.




Example without batching

In [ ]:
num_stories = 10
content = "Once upon a time,"

# serial example, with one story completion per request
for _ in range(num_stories):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": content}],
        max_tokens=20,
    )

    print(content + response.choices[0].message.content)

Example with batching multiple prompts with structured outputs

OpenAI's Structured Outputs feature offers a robust way to batch multiple prompts in a single request.

Here, rather than parsing raw text or hoping the model follows informal formatting, you specify a strict schema. This ensures your application can reliably parse the results by examining the defined structure. This eliminates the need for extensive validation or complicated parsing logic, as Structured Outputs guarantees consistent, type-safe data

In [ ]:
from pydantic import BaseModel

# Define pydantic model for structured output
class StoryResponse(BaseModel):
    stories: list[str]
    story_count: int

num_stories = 10
content = "Once upon a time,"

prompt_lines = [f"Story #{i+1}: {content}" for i in range(num_stories)]
prompt_text = "\n".join(prompt_lines)

messages = [
    {
        "role": "developer",
        "content": "You are a helpful assistant. Please respond to each prompt as a separate short story."
    },
    {
        "role": "user",
        "content": prompt_text
    }
]

# batched example, with all story completions in one request and using structured outputs
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=messages,
    response_format=StoryResponse,
)

print(response.choices[0].message.content)